In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Стили графиков
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [2]:
import numpy as np

def simulate_system(lambda_val, mu_val, tw_max, T_total):
    time = 0
    ts_prev = 0  # Время окончания обслуживания предыдущей заявки

    processed_count = 0
    lost_count = 0
    total_wait_time = 0
    busy_time = 0

    current_time = 0
    while current_time < T_total:
        # Генерируем время до следующей заявки (экспоненциальное)
        interarrival = np.random.exponential(1/lambda_val)
        current_time += interarrival

        # Генерируем допустимое время ожидания (равномерное [0, tw_max])
        dtw = np.random.uniform(0, tw_max)

        # Длительность обслуживания (экспоненциальное)
        service_duration = np.random.exponential(1/mu_val)

        # Логика событий (согласно рис 4.1 - 4.3)
        if current_time + dtw < ts_prev:
            lost_count += 1
        else:
            # Начало обслуживания
            start_service = max(current_time, ts_prev)
            wait_time = start_service - current_time
            total_wait_time += wait_time

            # Занятость канала (если простаивал до прихода)
            if ts_prev < current_time:
                busy_time += service_duration
            else:
                busy_time += service_duration

            ts_prev = start_service + service_duration
            processed_count += 1

    total_calls = processed_count + lost_count
    prob_process = processed_count / total_calls if total_calls > 0 else 0
    avg_wait = total_wait_time / processed_count if processed_count > 0 else 0
    load_factor = busy_time / T_total
    
    return {
        "processed": processed_count,
        "lost": lost_count,
        "prob_process": prob_process,
        "avg_wait": avg_wait,
        "load": load_factor
    }

In [3]:
print(simulate_system(lambda_val=0.7, mu_val=0.8, tw_max=1.0, T_total=1000))

{'processed': 425, 'lost': 273, 'prob_process': 0.6088825214899714, 'avg_wait': 0.0889605787744297, 'load': 0.538007472783581}
